# Sistema de Recomendação de Hotéis — Documentação do Modelo

**Introdução à Inteligência Artificial — Trabalho 1**

**Integrantes**
- Artur Kohara Guerra — matrícula 231025181  
- Celio Junio de Freitas Eduardo — matrícula 211010350  
- Rafael De Lima Pereira — matrícula 242043277  

Este notebook documenta a engenharia do modelo iterativo, detalhando a transição de um modelo de Feedback Implícito para um **Espaço Vetorial de Feedback Explícito Multicritério (10 Dimensões)** com ponderação **TF-IDF**. Execute as células **Python** na ordem (Python 3.10+; `pip install -r requirements.txt`).

> **Aviso:** A interface web (Streamlit) não deve ser iniciada dentro do Jupyter — ela roda num terminal à parte (`streamlit run app.py`).

## 1. Arquitetura de Dados: Espaço Vetorial de 10 Dimensões

Para contornar as limitações de agregação de informações escalares, o modelo mapeia hotéis e percepções de usuários em $\mathbf{v} \in \mathbb{R}^{10}$.
Os atributos analisados são: `luxo`, `lazer`, `urbano`, `pet_friendly`, `kids_friendly`, `acessibilidade`, `seguranca`, `preco`, `silencio`, e `capacidade`.

### 1.1 Ponderação TF-IDF (Term Frequency-Inverse Document Frequency)
Para aumentar a precisão, implementamos uma camada de ponderação que penaliza características genéricas e amplifica atributos distintivos do catálogo. 

- **TF (Frequência do Termo)**: Representada diretamente pelo valor da feature no hotel $f_{h,i} \in [0, 1]$.
- **DF (Frequência no Documento)**: Quantidade de hotéis onde a característica $i$ está presente (valor acima da média do catálogo).
- **IDF (Frequência Inversa)**: 
$$IDF_i = \log\left(\frac{N}{DF_i + 1}\right) + 1$$

O vetor final de pesos $\mathbf{w}_{IDF}$ é normalizado pelo seu valor máximo para manter a estabilidade numérica na escala de utilidade.

In [ ]:
import numpy as np
import pandas as pd
import math

FEATURES = ['luxo', 'lazer', 'urbano', 'pet_friendly', 'kids_friendly', 'acessibilidade', 'seguranca', 'preco', 'silencio', 'capacidade']
np.random.seed(42)

# --- Simulação de Cálculo IDF (recomendacao_controller.py) ---
def compute_idf_weights(matriz_hoteis):
    N = len(matriz_hoteis)
    col_means = matriz_hoteis.mean(axis=0)
    # DF: hotéis com feature acima da média (+1 suavização)
    df_counts = (matriz_hoteis > col_means).sum(axis=0) + 1
    idf = np.log(N / df_counts) + 1.0
    return idf / idf.max()

# Gerando catálogo simulado para cálculo de IDF
catalogo_simulado = np.random.beta(2, 5, (100, 10)) # A maioria dos hotéis tem valores baixos
idf_weights = compute_idf_weights(catalogo_simulado)

print("Pesos IDF calculados (Features raras ganham peso próximo a 1.0):")
print({f: round(val, 3) for f, val in zip(FEATURES, idf_weights)})

# --- Geração de Percepção Multicritério Sintética ---
def gerar_ratings_sinteticos(features_hotel_01):
    ruido_percepcao = np.random.normal(0, 0.5, size=len(FEATURES))
    ratings_vector = np.clip(np.round(features_hotel_01 * 4.0 + 1.0 + ruido_percepcao), 1, 5)
    return ratings_vector

hotel_exemplo = np.array([0.9, 0.4, 0.2, 0.1, 0.1, 0.5, 0.8, 0.1, 0.9, 0.3])
ratings_sinteticos = gerar_ratings_sinteticos(hotel_exemplo)

print("\nRatings gerados pelo usuário sintético (Escala [1, 5]):")
print({f: int(val) for f, val in zip(FEATURES, ratings_sinteticos)})

## 2. Perfil do Usuário e Vetor Final de Busca

Antes de executar qualquer algoritmo de recomendação, o sistema constrói o **vetor de busca** em três etapas:

### 2.1 Vetor de Contexto (`_build_context_vector`)
Mapeia as respostas do formulário (tipo de viagem, região, necessidades especiais) diretamente para $\mathbf{v}_{ctx} \in [0, 1]^{10}$. Sem histórico, esse vetor é usado isoladamente.

### 2.2 Perfil Latente do Usuário (`_build_user_profile`)
Dado o histórico de avaliações do usuário (10 colunas de notas em $[1, 5]$), o perfil é calculado em duas etapas:
1. **Normalização**: $w_i = \frac{nota_i - 1}{4}$ → projeta cada nota para $[0, 1]$
2. **Agregação**: $\mathbf{w}_{perfil} = \text{nanmean}(W_{\text{histórico}}, \text{axis}=0)$ → vetor médio das interações

### 2.3 Alpha-Blending (Fusão Contexto + Perfil)
O vetor final que alimenta KNN e FM combina os dois com peso $\alpha = 0.85$, priorizando o contexto imediato mas preservando o perfil histórico:

$$\mathbf{v}_{final} = 0.85 \cdot \mathbf{v}_{ctx} + 0.15 \cdot \mathbf{w}_{perfil}$$

In [ ]:
# --- Replicação exata de _build_user_profile ---

def build_user_profile(historico_notas_df):
    matriz_notas = historico_notas_df.values.astype(float)
    matriz_normalizada = (matriz_notas - 1.0) / 4.0
    perfil_medio = np.nanmean(matriz_normalizada, axis=0)
    return np.nan_to_num(perfil_medio, nan=0.1)

historico_simulado = pd.DataFrame([
    [5, 3, 1, 1, 1, 3, 5, 1, 5, 2],
    [4, 2, 1, 1, 1, 4, 4, 1, 4, 2],
    [5, 3, 2, 1, 1, 3, 5, 1, 5, 3],
], columns=FEATURES)

perfil_usuario = build_user_profile(historico_simulado)
ALPHA = 0.85
vetor_contexto_negocios = np.array([0.1, 0.0, 1.0, 0.1, 0.1, 0.1, 0.1, 0.8, 0.9, 0.1])

vetor_final = ALPHA * vetor_contexto_negocios + (1 - ALPHA) * perfil_usuario
print("Vetor final de busca (alpha-blending, alpha=0.85) [0, 1]:")
print({f: round(v, 3) for f, v in zip(FEATURES, vetor_final)})

## 3. Algoritmos de Ordenação: KNN e FM com TF-IDF

A controladora opera dois algoritmos de ranqueamento. Ambos projetam os vetores de entrada (busca e hotel) no espaço ponderado pelo **Vetor IDF** ($\mathbf{w}_{IDF}$) antes do cálculo de utilidade.

### 3.1 K-Nearest Neighbors (`_computar_knn`)
Mede a **similaridade de cosseno** ponderada:

$$\text{sim}(\mathbf{v}_{final}, \mathbf{f}_h) = \frac{(\mathbf{v}_{final} \odot \mathbf{w}_{IDF}) \cdot (\mathbf{f}_h \odot \mathbf{w}_{IDF})}{\|\mathbf{v}_{final} \odot \mathbf{w}_{IDF}\| \cdot \|\mathbf{f}_h \odot \mathbf{w}_{IDF}\| + \varepsilon}$$

Onde $\odot$ representa o produto de Hadamard (elemento a elemento). 

### 3.2 Factorization Machines (`_computar_fm`)
Estima a **utilidade bruta** via produto escalar ponderado e penaliza interações latentes de conflito:

$$\text{score}_{FM}(h) = (\mathbf{v}_{final} \odot \mathbf{w}_{IDF}) \cdot (\mathbf{f}_h \odot \mathbf{w}_{IDF}) - \lambda \cdot f_h^{\text{luxo}} \cdot f_h^{\text{urbano}}$$

onde $\lambda = 0.6$ é a penalidade do conflito *Luxo × Urbano*. Note que a penalidade é aplicada sobre os valores originais de conteúdo, enquanto a utilidade base é ponderada pela raridade das features.

In [ ]:
# --- Replicação exata de _computar_knn e _computar_fm (incluindo TF-IDF) ---

np.random.seed(7)
N_HOTEIS = 6
matriz_hoteis = np.random.beta(2, 2, (N_HOTEIS, 10))
ids_hoteis = [f'H{i:02d}' for i in range(N_HOTEIS)]

def computar_knn_tfidf(vetor_busca, matriz_h, weights, ids, offset=0, limit=3):
    # Aplicação do IDF antes do cálculo
    hotel_vectors_tfidf = matriz_h * weights
    final_vector_tfidf  = vetor_busca * weights

    dot_products = np.dot(hotel_vectors_tfidf, final_vector_tfidf)
    norm_hoteis  = np.linalg.norm(hotel_vectors_tfidf, axis=1)
    norm_busca   = np.linalg.norm(final_vector_tfidf)
    similaridades = dot_products / (norm_hoteis * norm_busca + 1e-9)

    indices_ordenados = np.argsort(similaridades)[::-1]
    pagina = indices_ordenados[offset : offset + limit]
    return [(ids[i], round(similaridades[i], 4)) for i in pagina]

def computar_fm_tfidf(vetor_busca, matriz_h, weights, ids, lambda_p=0.6, offset=0, limit=3):
    # Aplicação do IDF no produto escalar base
    matriz_tfidf       = matriz_h * weights
    final_vector_tfidf = vetor_busca * weights
    
    scores_base  = np.dot(matriz_tfidf, final_vector_tfidf)
    penalidades  = lambda_p * (matriz_h[:, 0] * matriz_h[:, 2]) 
    scores_finais = scores_base - penalidades

    indices_ordenados = np.argsort(scores_finais)[::-1]
    pagina = indices_ordenados[offset : offset + limit]
    return [(ids[i], round(scores_finais[i], 4)) for i in pagina]

print("KNN (TF-IDF) — Top 3:")
for h_id, s in computar_knn_tfidf(vetor_final, matriz_hoteis, idf_weights, ids_hoteis):
    print(f"  {h_id}: cosine_sim = {s}")

print("\nFM (TF-IDF) — Top 3:")
for h_id, s in computar_fm_tfidf(vetor_final, matriz_hoteis, idf_weights, ids_hoteis):
    print(f"  {h_id}: fm_score = {s}")

## 4. Avaliação de Performance (Métricas Globais)

As métricas são calculadas ao fim de cada sessão consumindo o histórico completo de avaliações.

### 4.1 RMSE — Avalia o FM
Para cada avaliação, computa a **predição real do FM** via `_obter_predicao_fm_real`:

1. **Produto Escalar**: $dot = \mathbf{w}_{usuario} \cdot \mathbf{f}_{hotel}$
2. **Normalização Dimensional**: $u = dot / 10$ (impede a explosão do valor em 10 dimensões)
3. **Reprojeção**: $\hat{y} = \text{clip}((u \times 4) + 1, 1, 5)$

$$\text{RMSE} = \sqrt{\frac{1}{M}\sum_{i=1}^{M}(y_i - \hat{y}_i)^2}$$

### 4.2 NDCG — Avalia o KNN
Calculado apenas sobre interações orgânicas da UI (`logica_geracao = 'KNN'`).

$$DCG_i = \frac{1}{\log_2(p_i + 1)}, \quad IDCG = 1.0, \quad NDCG_{global} = \frac{1}{|KNN|}\sum_i NDCG_i$$

In [ ]:
# --- Predição real para RMSE (recomendacao_controller.py) ---

def obter_predicao_fm_real(w_usuario_01, f_hotel_01):
    dot_product   = np.dot(w_usuario_01, f_hotel_01)   
    utilidade_norm = dot_product / 10.0                 # Normalização dimensional essencial
    predicao_escala = (utilidade_norm * 4.0) + 1.0      
    return float(np.clip(predicao_escala, 1.0, 5.0))

notas_fm = np.array([[5, 3, 1, 1, 1, 3, 5, 1, 5, 2], [4, 4, 2, 2, 1, 3, 4, 1, 4, 2]])
y_real = notas_fm.mean(axis=1)  
w_usuarios = (notas_fm - 1.0) / 4.0

erros_sq = []
for i in range(len(notas_fm)):
    pred = obter_predicao_fm_real(w_usuarios[i], hotel_exemplo)
    erros_sq.append((y_real[i] - pred) ** 2)

rmse = math.sqrt(np.mean(erros_sq))
print(f"RMSE Global (FM) Re-calculado: {rmse:.4f}")

## 5. Normalização de Scores para Exibição (UI)

Os scores brutos são heterogêneos e normalizados para a escala $[1, 5]$ na UI:

- **KNN**: `afinidade = ((s_max - score) / (s_max - s_min)) * 4 + 1`  
- **FM**: `afinidade = ((score - s_min) / (s_max - s_min)) * 4 + 1`

### Referências Acadêmicas
- **Salton & Buckley (1988)**: "Term-weighting approaches in automatic text retrieval". Information Processing & Management (Base para a implementação do TF-IDF sobre features do catálogo).
- **Pazzani & Billsus (2007)**: Sistemas Baseados em Conteúdo (Vetorização de atributos).
- **Rendle (2010)**: Factorization Machines (Interações latentes).
- **Järvelin & Kekäläinen (2002)**: Avaliação Cumulativa (DCG e NDCG).